# 01 - Generate synthetic raw data (FreshMart grocery chain)

**Business problem:** FreshMart (fictional 40-store grocery chain) loses ~8-9% of demand to on-shelf stockouts. The legacy min/max replenishment rule ignores supplier lead-time variability and promotions.

This notebook simulates 6 months (2026-05-01 to 2026-10-31) of store x SKU operations and drops **raw files** into the UC volume `/Volumes/serverless_stable_zlgeaw_catalog/stockout_raw/landing`, mimicking source systems:

| Folder | Source system | Format | Grain |
| --- | --- | --- | --- |
| `master/` | MDM | CSV | stores, products, suppliers |
| `pos_sales/` | POS (NCR) | CSV, one file per day | store x SKU x day |
| `inventory/` | WMS | JSON lines, one file per day | EOD store x SKU snapshot |
| `receipts/` | WMS receiving | JSON lines | PO receipts |
| `promotions/` | Merchandising | CSV | promo calendar |

Intentional data-quality defects (duplicates, negative units, null SKUs) are injected so the Lakeflow pipeline's expectations have something to catch. Store manager name/email/phone are synthetic PII used to demonstrate Unity Catalog masking. Lost demand (censored by stockouts) is **not** shipped in the raw files - like reality, it must be inferred.

In [0]:
import numpy as np, pandas as pd, os, shutil, datetime as dt
CATALOG = "serverless_stable_zlgeaw_catalog"
VOL = f"/Volumes/{CATALOG}/stockout_raw/landing"
START, END = dt.date(2026, 5, 1), dt.date(2026, 10, 31)
ROP_BUFFER_DAYS = 3.0   # legacy policy: reorder point = avg demand x (mean lead time + buffer)
rng = np.random.default_rng(42)
for d in ["master", "pos_sales", "inventory", "receipts", "promotions"]:
    p = f"{VOL}/{d}"
    if os.path.exists(p): shutil.rmtree(p)
    os.makedirs(p, exist_ok=True)

N_STORES, N_SKUS, N_SUP = 40, 150, 10
regions = ["Northeast", "Southeast", "Midwest", "Southwest", "West"]
formats = rng.choice(["Supercenter", "Neighborhood", "Express"], N_STORES, p=[.3, .5, .2])
fmt_factor = {"Supercenter": 1.8, "Neighborhood": 1.0, "Express": 0.55}
first = ["Ana","Ben","Chloe","Dev","Elena","Femi","Grace","Hiro","Isla","Jon","Kara","Luis","Maya","Nikhil","Olga","Priya","Quinn","Raj","Sara","Tom"]
last = ["Lopez","Nguyen","Smith","Okafor","Patel","Kim","Garcia","Brown","Rossi","Cohen"]
stores = pd.DataFrame({
  "store_id": [f"S{i:03d}" for i in range(1, N_STORES+1)],
  "store_name": [f"FreshMart #{i:03d}" for i in range(1, N_STORES+1)],
  "region": rng.choice(regions, N_STORES), "format": formats,
  "sq_ft": [int(rng.normal({"Supercenter":120000,"Neighborhood":45000,"Express":12000}[f], 3000)) for f in formats],
  "open_date": [str(dt.date(2010,1,1)+dt.timedelta(days=int(x))) for x in rng.integers(0, 5000, N_STORES)]})
names = [f"{rng.choice(first)} {rng.choice(last)}" for _ in range(N_STORES)]
stores["store_manager"] = names
stores["manager_email"] = [n.lower().replace(" ", ".")+f"{i}@freshmart-example.com" for i, n in enumerate(names)]
stores["manager_phone"] = [f"555-{rng.integers(100,999)}-{rng.integers(1000,9999)}" for _ in range(N_STORES)]

suppliers = pd.DataFrame({
  "supplier_id": [f"V{i:02d}" for i in range(1, N_SUP+1)],
  "supplier_name": ["Valley Dairy Co","SunGrow Produce","Crunch Snacks Inc","Bubbly Beverages","CleanHome Goods","PureCare Labs","Golden Bakery","Harbor Frozen","Prairie Pantry","Metro Distributors"],
  "lead_time_days_mean": [2,1,5,4,7,6,1,4,5,3],
  "lead_time_days_sd": [0.5,0.4,1.5,1.2,2.5,2.0,0.3,1.5,1.8,1.0],
  "fill_rate": [0.97,0.92,0.90,0.95,0.86,0.88,0.98,0.91,0.93,0.94]})
cats = {"Dairy":("V01",(2,6),True), "Produce":("V02",(1,5),True), "Snacks":("V03",(2,7),False), "Beverages":("V04",(1,4),False),
        "Household":("V05",(4,15),False), "Personal Care":("V06",(3,12),False), "Bakery":("V07",(2,6),True), "Frozen":("V08",(3,9),False),
        "Pantry":("V09",(1,8),False)}
cat_list = list(cats); prod_rows = []
for i in range(1, N_SKUS+1):
    c = cat_list[(i-1) % len(cat_list)]; sup, (lo, hi), per = cats[c]
    if c in ("Pantry", "Snacks") and rng.random() < .3: sup = "V10"
    price = round(rng.uniform(lo, hi), 2)
    prod_rows.append({"sku_id": f"SKU{i:04d}", "sku_name": f"{c} item {i:03d}", "category": c, "supplier_id": sup,
       "unit_price": price, "unit_cost": round(price*rng.uniform(.55,.75), 2), "case_pack": int(rng.choice([6,12,24])),
       "is_perishable": per, "shelf_capacity": 0})
products = pd.DataFrame(prod_rows)
print(len(stores), "stores |", len(products), "SKUs |", len(suppliers), "suppliers")

In [0]:
dates = pd.date_range(START, END, freq="D"); D = len(dates); S, K = N_STORES, N_SKUS
ffac = np.array([fmt_factor[f] for f in formats])[:, None]
base = rng.lognormal(mean=1.3, sigma=0.7, size=(1, K)) * ffac * rng.uniform(.8, 1.2, (S, 1))
products["shelf_capacity"] = np.maximum(products.case_pack*2, (base.mean(0)*14/products.case_pack).round()*products.case_pack).astype(int)
cap = np.maximum(np.tile(products.shelf_capacity.values, (S, 1)).astype(float)*ffac, products.case_pack.values*2)
dow = np.array([0.9, 0.85, 0.9, 1.0, 1.2, 1.4, 1.1])  # Mon..Sun
sup_idx = products.supplier_id.map({s: i for i, s in enumerate(suppliers.supplier_id)}).values
lt_mean, lt_sd, fill = (suppliers[c].values[sup_idx] for c in ["lead_time_days_mean", "lead_time_days_sd", "fill_rate"])

# promotions: 1-3 windows per SKU, chain-wide or regional
promo_rows = []; promo_lift = np.ones((D, S, K)); store_region = stores.region.values
for k in range(K):
    for _ in range(rng.integers(1, 4)):
        st = rng.integers(0, D-14); ln = int(rng.choice([7, 10, 14])); lift = round(rng.uniform(1.5, 2.6), 2)
        reg = "ALL" if rng.random() < .5 else rng.choice(regions)
        mask = np.ones(S, bool) if reg == "ALL" else (store_region == reg)
        promo_lift[st:st+ln][:, mask, k] *= lift
        promo_rows.append({"promo_id": f"P{len(promo_rows)+1:04d}", "sku_id": products.sku_id[k], "region": reg,
          "start_date": str(dates[st].date()), "end_date": str(dates[min(st+ln-1, D-1)].date()),
          "promo_type": rng.choice(["TPR", "BOGO", "Circular"]), "expected_lift": lift})
promos = pd.DataFrame(promo_rows)

# legacy min/max policy: no lead-time variability, no promo awareness
rop = base * (lt_mean[None, :] + ROP_BUFFER_DAYS)
on_hand = np.round(cap*0.7); pipeline = []; open_order = np.zeros((S, K), bool); po_seq = 0; stats = []
ss, kk = np.meshgrid(np.arange(S), np.arange(K), indexing="ij")
sid, kid = stores.store_id.values[ss.ravel()], products.sku_id.values[kk.ravel()]
trend = 1 + 0.0008*np.arange(D)
for t, d in enumerate(dates):
    arrivals = [p for p in pipeline if p[0] == t]; pipeline = [p for p in pipeline if p[0] != t]; rec_today = []
    for a, s, k, q, po, oi, ei, oq in arrivals:
        on_hand[s, k] += q; open_order[s, k] = False
        rec_today.append({"po_id": po, "store_id": stores.store_id[s], "sku_id": products.sku_id[k], "order_date": str(dates[oi].date()),
          "expected_date": str((dates[oi]+pd.Timedelta(days=int(ei))).date()), "received_date": str(d.date()),
          "ordered_units": int(oq), "received_units": int(q)})
    lam = base*dow[d.dayofweek]*promo_lift[t]*trend[t]*rng.gamma(8, 1/8, (S, K))
    demand = rng.poisson(lam); sold = np.minimum(demand, on_hand).astype(int); on_hand -= sold
    shrink = (rng.random((S, K)) < 0.01)*rng.integers(1, 4, (S, K)); on_hand = np.maximum(on_hand-shrink, 0)
    for s, k in zip(*np.nonzero((on_hand <= rop) & (~open_order))):
        oq = int(np.ceil(max(cap[s, k]-on_hand[s, k], 1)/products.case_pack[k])*products.case_pack[k])
        lt = max(1, int(round(rng.normal(lt_mean[k], lt_sd[k]))))
        rq = int(np.floor(oq*min(1, rng.normal(fill[k], 0.05))/products.case_pack[k])*products.case_pack[k]) if rng.random() > 0.03 else 0
        po_seq += 1; pipeline.append((t+lt, s, k, rq, f"PO{po_seq:07d}", t, lt_mean[k], oq)); open_order[s, k] = True
    on_order = np.zeros((S, K))
    for a, s, k, q, po, oi, ei, oq in pipeline: on_order[s, k] += oq
    # POS file (only rows with sales, plus injected defects)
    df = pd.DataFrame({"store_id": sid, "sku_id": kid, "business_date": str(d.date()), "units_sold": sold.ravel()})
    df["net_sales"] = (df.units_sold*np.tile(products.unit_price.values, S)*np.where(promo_lift[t].ravel() > 1, 0.8, 1.0)).round(2)
    df["txn_count"] = np.maximum((df.units_sold*rng.uniform(.6, .95, len(df))).round(), 0).astype(int)
    df = df[df.units_sold > 0].copy()
    df = pd.concat([df, df.sample(frac=0.002, random_state=t), df.sample(frac=0.001, random_state=t+1000).assign(units_sold=-1),
                    df.sample(n=3, random_state=t+2000).assign(sku_id=None)])
    df["pos_system"] = "NCR-Emerald"; df["file_ts"] = f"{d.date()}T23:55:00Z"
    df.to_csv(f"{VOL}/pos_sales/pos_sales_{d:%Y%m%d}.csv", index=False)
    pd.DataFrame({"store_id": sid, "sku_id": kid, "snapshot_date": str(d.date()), "on_hand_units": on_hand.ravel().astype(int),
        "on_order_units": on_order.ravel().astype(int), "snapshot_ts": f"{(d+pd.Timedelta(days=1)).date()}T02:00:00Z",
        "source": "WMS-Manhattan"}).to_json(f"{VOL}/inventory/inventory_{d:%Y%m%d}.json", orient="records", lines=True)
    if rec_today: pd.DataFrame(rec_today).to_json(f"{VOL}/receipts/receipts_{d:%Y%m%d}.json", orient="records", lines=True)
    stats.append((demand.sum(), sold.sum(), (on_hand == 0).mean(), ((demand-sold)*products.unit_price.values).sum()))

stores.to_csv(f"{VOL}/master/stores.csv", index=False); products.to_csv(f"{VOL}/master/products.csv", index=False)
suppliers.to_csv(f"{VOL}/master/suppliers.csv", index=False); promos.to_csv(f"{VOL}/promotions/promotions.csv", index=False)

In [0]:
so = np.array(stats)
print(f"days={D} store_sku_pairs={S*K} purchase_orders={po_seq} promos={len(promos)}")
print(f"true demand={so[:,0].sum():,.0f} units | sold={so[:,1].sum():,.0f} | lost={so[:,0].sum()-so[:,1].sum():,.0f} ({1-so[:,1].sum()/so[:,0].sum():.1%})")
print(f"avg end-of-day zero-stock rate={so[:,2].mean():.2%}")
print(f"lost revenue, 6 months, 40 stores = ${so[:,3].sum():,.0f}  -> annualized ${so[:,3].sum()*365/D:,.0f}")
for d in ["master", "pos_sales", "inventory", "receipts", "promotions"]: print(f"{d:<11} {len(os.listdir(f'{VOL}/{d}'))} files")